# Generación de Descripciones Tácticas con LLM

Este notebook implementa un sistema de generación de descripciones tácticas de secuencias de posesión usando un LLM local, con 3 evoluciones progresivas del input:

1. **Evolución 1**: Solo eventos
2. **Evolución 2**: Eventos + Documentación (RAG)
3. **Evolución 3**: Eventos + Documentación + Formaciones + Roles

## 1. Setup e Importaciones

Importamos las librerías necesarias para el procesamiento de datos, modelos de lenguaje y análisis táctico.

In [7]:
import os
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from accelerate import Accelerator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("✅ Librerías importadas correctamente")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo CUDA: {torch.cuda.get_device_name(0)}")

✅ Librerías importadas correctamente
PyTorch version: 2.10.0+cpu
CUDA disponible: False


## 2. Carga de Datos

Cargamos los datos de tracking y eventos del Sample Game 1.

In [2]:
print("Cargando datos...")
tracking_home = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Home_Team.csv', header=2)
tracking_away = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawTrackingData_Away_Team.csv', header=2)
events = pd.read_csv('../../data/metrica/Sample_Game_1/Sample_Game_1_RawEventsData.csv')

print(f"✅ Tracking Home: {len(tracking_home)} frames")
print(f"✅ Tracking Away: {len(tracking_away)} frames")
print(f"✅ Eventos: {len(events)} eventos")

Cargando datos...
✅ Tracking Home: 145006 frames
✅ Tracking Away: 145006 frames
✅ Eventos: 1745 eventos


## 3. Función build_pro_tactical_phases

Importamos la función del notebook existente para identificar secuencias de posesión.

In [3]:
def get_attacking_direction_by_team(tracking_home, tracking_away):
    """
    Define la dirección de ataque para cada equipo y cada periodo (tiempo),
    basándose en la posición del jugador más cercano a la línea de fondo izquierda (x=0)
    y derecha (x=1) en el kickoff de cada periodo.

    Retorna un diccionario:
        {
            'Home': {1: 'left-to-right' o 'right-to-left', 2: ...},
            'Away': {1: ..., 2: ...}
        }
    """
    directions = {'Home': {}, 'Away': {}}

    for period in [1, 2]:
        # Filtramos los datos para el periodo correspondiente
        tracking_home_p = tracking_home[tracking_home['Period'] == period]
        tracking_away_p = tracking_away[tracking_away['Period'] == period]

        player_cols_home = [c for c in tracking_home.columns if c.startswith('Player') and 'Unnamed' not in c ]
        player_cols_away = [c for c in tracking_away.columns if c.startswith('Player') and 'Unnamed' not in c ]

        # Tomamos la primera fila del periodo (ej. inicio/kickoff)
        if len(tracking_home_p) == 0 or len(tracking_away_p) == 0:
            print(f"Periodo {period} vacío para Home o Away. Saltando.")
            continue

        home_kickoff = tracking_home_p.iloc[0]
        away_kickoff = tracking_away_p.iloc[0]
        
        # Extraemos las posiciones x de todos los jugadores en el kickoff
        home_x_positions = home_kickoff[player_cols_home].dropna().values
        away_x_positions = away_kickoff[player_cols_away].dropna().values

        # Encontramos la posición mínima (más cerca de x=0, línea de fondo izquierda) y máxima (x=1, derecha)
        home_min_x = home_x_positions.min()
        home_max_x = home_x_positions.max()
        away_min_x = away_x_positions.min()
        away_max_x = away_x_positions.max()

        # Decisión:
        # Si la mayoría de los jugadores están en la mitad izquierda (x < 0.5), asumimos que atacan hacia la derecha
        # (left-to-right). Caso contrario hacia la izquierda.
        # Nota: Se puede sofisticar usando la posición promedio o la dispersión.

        # Equipo local
        if np.mean(home_x_positions) < 0.5:
            # Los jugadores están sobre la izquierda, atacan hacia la derecha
            directions['Home'][period] = 1
        else:
            # Los jugadores están sobre la derecha, atacan hacia la izquierda
            directions['Home'][period] = -1

        # Equipo visitante
        if np.mean(away_x_positions) < 0.5:
            directions['Away'][period] = 1
        else:
            directions['Away'][period] = -1

    return directions

In [4]:
directions = get_attacking_direction_by_team(
    tracking_home,
    tracking_away
)

# Aggiungi la direzione di attacco agli eventi
events['Attacking_Direction'] = events.apply(
    lambda row: directions[row['Team']][row['Period']], 
    axis=1
)

In [5]:
def build_pro_tactical_phases(events_df):
    """
    Identifica secuencias de posesión (fases tácticas) a partir de eventos.
    
    Args:
        events_df: DataFrame con eventos del partido
        
    Returns:
        DataFrame con fases tácticas identificadas
    """
    # Trabajar sobre una copia para no alterar el original
    df = events_df.copy()
    
    # ---------------------------------------------------------
    # 1. LÓGICA DE POSESIÓN (Algoritmo de Dueño)
    # ---------------------------------------------------------
    possession_owners = []
    # Inicializar con el primer equipo que hace algo
    current_owner = df.iloc[0]['Team']
    
    # Eventos que NO cambian la posesión (Interrupciones)
    interruptions = ['FAULT RECEIVED', 'CARD', 'BALL OUT', 'CHALLENGE', 'RECOVERY']
    
    for idx, row in df.iterrows():
        evt_type = row['Type']
        evt_sub = str(row['Subtype'])
        evt_team = row['Team']
        
        # Un regate ganado CONFIRMA la posesión
        if evt_type == 'DRIBBLE' and 'WON' in evt_sub:
            current_owner = evt_team
        # Balón parado INICIA posesión
        elif evt_type == 'SET PIECE':
            current_owner = evt_team
        # Acciones activas CONFIRMAN posesión
        elif evt_type in ['PASS', 'SHOT', 'TOQUE']:
            current_owner = evt_team
        # Si es interrupción, mantenemos el dueño anterior
        elif evt_type in interruptions:
            pass 
        
        possession_owners.append(current_owner)
    
    df['Smart_Owner'] = possession_owners
    
    # Detectar CAMBIO DE FASE:
    # 1. Cambia el equipo dueño.
    # 2. Ocurre un Balón Parado (Set Piece) -> Reinicia jugada.
    df['New_Phase'] = (df['Smart_Owner'] != df['Smart_Owner'].shift()) | (df['Type'] == 'SET PIECE')
    df['Phase_ID'] = df['New_Phase'].cumsum()
    
    # ---------------------------------------------------------
    # 2. AGREGACIÓN (Micro-Ciclos)
    # ---------------------------------------------------------
    # Agrupamos los eventos por Phase_ID para sacar métricas de la jugada completa
    phases = df.groupby('Phase_ID').agg(
        Team=('Smart_Owner', 'first'),
        Period=('Period', 'first'),
        Start_Time=('Start Time [s]', 'min'),
        Duration=('Start Time [s]', lambda x: x.max() - x.min()),
        Event_Count=('Type', 'count'),
        # IMPORTANTE: Usamos nombres con guion bajo para estandarizar
        Start_Frame=('Start Frame', 'min'), 
        End_Frame=('End Frame', 'max'),     
        Start_X=('Start X', 'first'), 
        End_X=('End X', 'last'),    
        Events_List=('Type', list),
        Subtypes_List=('Subtype', list),
        Events_Ids = ('Type', lambda x: list(x.index)),
        Start_Type=('Type', 'first'),
        Start_Subtype=('Subtype', 'first')
    ).reset_index()
    
    # ---------------------------------------------------------
    # 3. ENRIQUECIMIENTO TÁCTICO (Zonas y Resultado)
    # ---------------------------------------------------------
    
    # Función de Zona (Normalizada 0-1)
    def get_zone(x_coord, period, team):
        # Lógica Metrica: Home ataca a 1 en P1, a 0 en P2
        attack_dir = 1 
        if (team == 'Home' and period == 2) or (team == 'Away' and period == 1):
            attack_dir = -1 
            
        # Normalizar X relativo a "Mi Portería" (0) -> "Rival" (1)
        rel_x = x_coord if attack_dir == 1 else (1.0 - x_coord)
            
        if rel_x < 0.35: return "Iniciación"
        if rel_x < 0.65: return "Creación"
        return "Finalización"

    # Aplicar Zonas
    phases['Start_Zone'] = phases.apply(lambda r: get_zone(r['Start_X'], r['Period'], r['Team']), axis=1)
    phases['End_Zone'] = phases.apply(lambda r: get_zone(r['End_X'], r['Period'], r['Team']), axis=1)
    
    # Definir Contexto (Cómo empezó)
    def define_context(row):
        if row['Start_Type'] == 'SET PIECE': return 'ABP'
        if 'RECOVERY' in row['Events_List']: return 'Recuperación'
        return 'Juego Abierto'
    phases['Context'] = phases.apply(define_context, axis=1)
    
    # Definir Resultado (Outcome)
    def define_outcome(row):
        evs = row['Events_List']
        subs = [str(x) for x in row['Subtypes_List']]
        
        if 'SHOT' in evs:
            if any('GOAL' in s for s in subs): return 'GOL'
            return 'Tiro'
        if 'BALL LOST' in evs: return 'Pérdida'
        if 'BALL OUT' in evs: return 'Fuera'
        return 'Posesión'

    phases['Outcome'] = phases.apply(define_outcome, axis=1)
    
    # Filtrar jugadas "basura" (menos de 1 segundo)
    return phases[phases['Duration'] > 1.0].copy()

# Generar fases tácticas
print("⚙️ Generando Fases Tácticas (pro_phases)...")
pro_phases = build_pro_tactical_phases(events)
print(f"✅ Variable 'pro_phases' definida con {len(pro_phases)} jugadas.")
print("Columnas:", pro_phases.columns.tolist())

⚙️ Generando Fases Tácticas (pro_phases)...
✅ Variable 'pro_phases' definida con 188 jugadas.
Columnas: ['Phase_ID', 'Team', 'Period', 'Start_Time', 'Duration', 'Event_Count', 'Start_Frame', 'End_Frame', 'Start_X', 'End_X', 'Events_List', 'Subtypes_List', 'Events_Ids', 'Start_Type', 'Start_Subtype', 'Start_Zone', 'End_Zone', 'Context', 'Outcome']


## 3.1. Funciones de Soporte

Funciones auxiliares para trabajar con eventos y fases tácticas.

In [6]:
def get_events_by_possession_id(phases_df, phase_id, events_df):
    """
    Recupera los eventos de una secuencia de posesión específica.
    
    Args:
        phases_df: DataFrame con fases tácticas (pro_phases)
        phase_id: ID de la fase a recuperar
        events_df: DataFrame con eventos del partido
        
    Returns:
        DataFrame con eventos de la fase, o None si no se encuentra
    """
    # Recupera la fila de la fase correspondiente
    phase_row = phases_df.loc[phases_df['Phase_ID'] == phase_id]
    if phase_row.empty:
        return None

    # Extrae los valores de Team y Events_Ids
    possession_team = phase_row.iloc[0]['Team']
    events_ids_list = phase_row.iloc[0]['Events_Ids']

    # events_ids_list dovrebbe essere una lista di indici o ID
    if not isinstance(events_ids_list, list):
        raise ValueError("Il campo 'Events_Ids' non contiene una lista.")

    # Devuelve las filas del DataFrame correspondientes a los eventos de la fase
    possession_events = events_df.loc[events_ids_list].copy()
    possession_events['Possession_Team'] = possession_team
    return possession_events


def format_events_as_text(events_list):
    """
    Convierte una lista de eventos (DataFrame o lista de diccionarios) en texto estructurado
    para usar en prompts de LLM. Añade la dirección de ataque por evento usando la columna 'Attacking_Direction' si está disponible.
    
    Args:
        events_list: DataFrame o lista de diccionarios con eventos
        
    Returns:
        String con eventos formateados en texto estructurado, con dirección de ataque incluida.
    """
    # Convertir a DataFrame si es necesario
    if isinstance(events_list, list):
        if len(events_list) == 0:
            return "No hay eventos disponibles."
        # Si es lista de diccionarios, convertir a DataFrame
        if isinstance(events_list[0], dict):
            events_df = pd.DataFrame(events_list)
        else:
            return "Formato de eventos no reconocido."
    elif isinstance(events_list, pd.DataFrame):
        events_df = events_list.copy()
    else:
        return "Formato de eventos no reconocido."
    
    if len(events_df) == 0:
        return "No hay eventos disponibles."
    
    # Ordenar por tiempo si existe la columna
    if 'Start Time [s]' in events_df.columns:
        events_df = events_df.sort_values('Start Time [s]')
    
    # Construir texto estructurado
    lines = []
    lines.append("=== SECUENCIA DE EVENTOS ===\n")
    
    for idx, (_, event) in enumerate(events_df.iterrows(), 1):
        event_parts = []
        
        # Número de evento
        event_parts.append(f"[Evento {idx}]")
        
        # Equipo
        team = event.get('Team', 'N/A')
        event_parts.append(f"Equipo: {team}")
        
        # Tipo y Subtipo
        event_type = event.get('Type', 'N/A')
        subtype = event.get('Subtype', None)
        if pd.notna(subtype) and str(subtype) != 'nan':
            event_parts.append(f"Tipo: {event_type} ({subtype})")
        else:
            event_parts.append(f"Tipo: {event_type}")
        
        # Tiempo
        start_time = event.get('Start Time [s]', None)
        if pd.notna(start_time):
            event_parts.append(f"Tiempo: {start_time:.2f}s")
        
        # Jugadores
        from_player = event.get('From', None)
        to_player = event.get('To', None)
        if pd.notna(from_player):
            if pd.notna(to_player):
                event_parts.append(f"Jugadores: {from_player} → {to_player}")
            else:
                event_parts.append(f"Jugador: {from_player}")
        
        # Ubicación
        start_x = event.get('Start X', None)
        start_y = event.get('Start Y', None)
        end_x = event.get('End X', None)
        end_y = event.get('End Y', None)
        
        location_parts = []
        if pd.notna(start_x) and pd.notna(start_y):
            location_parts.append(f"({start_x:.3f}, {start_y:.3f})")
        if pd.notna(end_x) and pd.notna(end_y) and (end_x != start_x or end_y != start_y):
            location_parts.append(f"→ ({end_x:.3f}, {end_y:.3f})")
        
        if location_parts:
            event_parts.append(f"Ubicación: {' '.join(location_parts)}")
        
        # Dirección de ataque desde la columna Attacking_Direction (en español)
        attacking_direction = event.get('Attacking_Direction', None)
        if pd.notna(attacking_direction):
            if attacking_direction == 1:
                dir_txt = "Ataque de izquierda a derecha (línea de fondo defensiva = x bajo, línea de fondo ofensiva = x alto, Attacking_Direction=1)"
            elif attacking_direction == -1:
                dir_txt = "Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1)"
            else:
                dir_txt = f"Dirección de ataque desconocida (Attacking_Direction={attacking_direction})"
            event_parts.append(f"Dirección de ataque: {dir_txt}")
        
        # Periodo
        period = event.get('Period', None)
        if pd.notna(period):
            event_parts.append(f"Periodo: {int(period)}")
        
        lines.append(" | ".join(event_parts))
    
    return "\n".join(lines)



# Test de las funciones
print("✅ Funciones de soporte definidas:")
print("  - get_events_by_possession_id")
print("  - format_events_as_text")
print("\n📝 Probando format_events_as_text con una secuencia de ejemplo...")

# Obtener eventos de la primera fase
test_events = get_events_by_possession_id(pro_phases, phase_id=1, events_df=events)
if test_events is not None:
    formatted_text = format_events_as_text(test_events.head(3))
    print("\nEjemplo de formato:")
    print(formatted_text)
else:
    print("No se pudo obtener eventos de prueba.")

✅ Funciones de soporte definidas:
  - get_events_by_possession_id
  - format_events_as_text

📝 Probando format_events_as_text con una secuencia de ejemplo...

Ejemplo de formato:
=== SECUENCIA DE EVENTOS ===

[Evento 1] | Equipo: Away | Tipo: SET PIECE (KICK OFF) | Tiempo: 0.04s | Jugador: Player19 | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1) | Periodo: 1
[Evento 2] | Equipo: Away | Tipo: PASS | Tiempo: 0.04s | Jugadores: Player19 → Player21 | Ubicación: (0.450, 0.390) → (0.550, 0.430) | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1) | Periodo: 1
[Evento 3] | Equipo: Away | Tipo: PASS | Tiempo: 0.12s | Jugadores: Player21 → Player15 | Ubicación: (0.550, 0.430) → (0.580, 0.210) | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva =

In [7]:
events.head()

,Team,Type,Subtype,Period,Start Frame,Start Time [s],End Frame,End Time [s],From,To,Start X,Start Y,End X,End Y,Attacking_Direction
0,Away,SET PIECE,KICK OFF,1,1,0.04,0,0.00,Player19,NaN,NaN,NaN,NaN,NaN,-1
1,Away,PASS,NaN,1,1,0.04,3,0.12,Player19,Player21,0.45,0.39,0.55,0.43,-1
2,Away,PASS,NaN,1,3,0.12,17,0.68,Player21,Player15,0.55,0.43,0.58,0.21,-1
3,Away,PASS,NaN,1,45,1.80,61,2.44,Player15,Player19,0.55,0.19,0.45,0.31,-1
4,Away,PASS,NaN,1,77,3.08,96,3.84,Player19,Player21,0.45,0.32,0.49,0.47,-1


In [8]:
print(format_events_as_text(test_events))

=== SECUENCIA DE EVENTOS ===

[Evento 1] | Equipo: Away | Tipo: SET PIECE (KICK OFF) | Tiempo: 0.04s | Jugador: Player19 | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1) | Periodo: 1
[Evento 2] | Equipo: Away | Tipo: PASS | Tiempo: 0.04s | Jugadores: Player19 → Player21 | Ubicación: (0.450, 0.390) → (0.550, 0.430) | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1) | Periodo: 1
[Evento 3] | Equipo: Away | Tipo: PASS | Tiempo: 0.12s | Jugadores: Player21 → Player15 | Ubicación: (0.550, 0.430) → (0.580, 0.210) | Dirección de ataque: Ataque de derecha a izquierda (línea de fondo defensiva = x alto, línea de fondo ofensiva = x bajo, Attacking_Direction=-1) | Periodo: 1
[Evento 4] | Equipo: Away | Tipo: PASS | Tiempo: 1.80s | Jugadores: Player15 → Player19 | Ubicación: (0.550, 0.190) → (0.450, 0.310) 

## 4. Carga de Documentación Táctica

Cargamos la documentación táctica que servirá como knowledge base para el RAG.

In [9]:
# Cargar documentación táctica
import re

doc_path = '../../Analisis_Tactico_Futbol_ES.md'

with open(doc_path, 'r', encoding='utf-8') as f:
    tactical_documentation = f.read()

print(f"✅ Documentación cargada: {len(tactical_documentation)} caracteres")
print(f"Primeras 200 caracteres:\n{tactical_documentation[:200]}...")


def chunk_documentation_by_sections(text):
    """
    Chunkea la documentación por secciones markdown (headers ## ).
    Cada chunk incluye el título de la sección y su contenido completo.
    Útil para RAG: recuperar secciones relevantes por similitud semántica.
    """
    chunks = []
    # Split por líneas que empiezan con ## (nivel 2)
    parts = re.split(r'\n(?=## )', text.strip())
    for part in parts:
        part = part.strip()
        if not part:
            continue
        lines = part.split('\n', 1)
        header = lines[0].strip()
        body = lines[1].strip() if len(lines) > 1 else ""
        # Título limpio (quitar ## y emojis opcionales)
        title_clean = re.sub(r'^#+\s*', '', header).strip()
        chunks.append({
            "title": title_clean,
            "text": part  # texto completo de la sección (header + body) para embedding/retrieval
        })
    return chunks


doc_chunks = chunk_documentation_by_sections(tactical_documentation)
print(f"\n✅ Documentación chunkada en {len(doc_chunks)} secciones.")
print("Títulos de las secciones:", [c["title"][:50] for c in doc_chunks])

✅ Documentación cargada: 32153 caracteres
Primeras 200 caracteres:
# Documentación de Análisis Táctico de Fútbol

**Guía Completa de Fases de Juego**

---

## 📋 Índice

1. [Introducción](#introducción)
2. [División del Campo](#división-del-campo)
3. [Fase Ofensiva](#...

✅ Documentación chunkada en 17 secciones.
Títulos de las secciones: ['Documentación de Análisis Táctico de Fútbol', '📋 Índice', 'Introducción', 'División del Campo', 'FASE OFENSIVA', '1. Construcción Baja (Zona 1)', '2. Amplitudes y Desarrollo (Zona 2)', '3. Zona de Refinamiento (Entre Zona 2 y Zona 3)', '4. Ataque de la Línea (Zona 3)', '5. Transiciones Ofensivas', 'FASE DEFENSIVA', '6. Primera Presión (Zona 3 rival)', '7. Bloque Medio (Zona 2 propia)', '8. Construcción de Duelos en Bandas', '9. Bloque Bajo (Zona 1 propia)', '10. Transiciones Defensivas', 'Glosario de Conceptos']


In [10]:
# Crear embeddings de la documentación con sentence-transformers (para RAG)
# Modelo multilingüe adecuado para documentación en español
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Cargando modelo de embeddings...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)

# Textos a embeddar: cada chunk completo (título + cuerpo) para retrieval semántico
chunk_texts = [c["text"] for c in doc_chunks]
doc_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    normalize_embeddings=True,
)

print(f"✅ Embeddings creados: shape {doc_embeddings.shape} (n_chunks={len(doc_chunks)}, dim={doc_embeddings.shape[1]})")

Cargando modelo de embeddings...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings creados: shape (17, 384) (n_chunks=17, dim=384)


## 5. Setup LLM

Cargamos el modelo de lenguaje Phi-3-mini-128k-instruct con su tokenizer y configuramos el modelo con `device_map="auto"` para distribución automática de memoria.

In [11]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

def generate_with_api(prompt, max_new_tokens=300, temperature=0.7):
    completion = client.chat.completions.create(
        model="Qwen/Qwen3-Coder-Next:novita",
        messages=[
            {"role": "system", "content": "Eres un analista táctico de fútbol profesional."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_new_tokens,
        temperature=temperature
    )
    return completion.choices[0].message.content.strip()


## 6. Evolución 1: Solo Eventos

Generación de descripciones tácticas usando únicamente la información de eventos de la secuencia de posesión.

In [ ]:
def generate_description_v1(events_list, x_min=0, x_max=1, y_min=0, y_max=1):
    """
    Genera una descripción táctica de una secuencia de posesión usando solo los eventos.
    
    Args:
        events_list: DataFrame o lista de diccionarios con eventos de la secuencia
        
    Returns:
        String con la descripción táctica generada
    """
    # Formatear eventos como texto estructurado
    events_text = format_events_as_text(events_list)
    
    # Construir prompt para el LLM
    prompt = f"""Eres un analista táctico de fútbol profesional. Analiza la siguiente secuencia de eventos de una jugada y genera una descripción táctica detallada en español.

{events_text}

Instrucciones:
- Sé conciso en la descripción.
- Describe el desarrollo de la jugada de forma narrativa y fluida.
- Identifica patrones tácticos (pases cortos, cambios de orientación, progresión, etc.).
- Menciona las zonas del campo donde ocurren las acciones principales.
- x={x_min} línea de fondo izquierda, x={x_max} línea de fondo derecha
- y={y_max} banda derecha (para quien ataca de izquierda a derecha), y={y_min} banda izquierda (para quien ataca de izquierda a derecha)
- Zona defensiva: x <= {x_max/3}, Zona central: {x_max/3} < x <= {2*x_max/3}, Zona ofensiva: x > {2*x_max/3} (para quien ataca de izquierda a derecha; y viceversa para quien ataca de derecha a izquierda) 
- Divide el campo en cinco franjas a lo largo del eje y (canales laterales y centrales): 
    - y < {y_max/5} pasillo izquierdo, 
    - {y_max/5} ≤ y < {2*y_max/5} interior izquierdo,
    - {2*y_max/5} ≤ y < {3*y_max/5} carril central,
    - {3*y_max/5} ≤ y < {4*y_max/5} interior derecho,
    - y ≥ {4*y_max/5} pasillo derecho
  (para quien ataca de izquierda a derecha; y viceversa si se ataca de derecha a izquierda)
- Describe el resultado final de la secuencia.
- Usa terminología táctica apropiada (ej: "construcción desde atrás", "transición", "finalización").
- Sé específico sobre los jugadores y sus acciones cuando sea relevante.
- No añadas suposiciones sobre roles de jugadores a menos que estén explícitamente mencionadas en los datos de eventos.


Descripción táctica:"""

    
    # Generar descripción usando el LLM
    description = generate_with_api(
        prompt,
        max_new_tokens=1000,
        temperature=0.7
    )
    
    return description.strip()


print("✅ Función generate_description_v1 definida")
print("📝 Usa generate_description_v1(events_list) para generar descripciones tácticas")

✅ Función generate_description_v1 definida
📝 Usa generate_description_v1(events_list) para generar descripciones tácticas


### 6.1. Prueba de Evolución 1

Probamos la generación de descripciones con algunas secuencias de ejemplo.

In [18]:
# Seleccionar algunas secuencias interesantes para probar
test_phase_ids = [4,]

print("🧪 Generando descripciones tácticas con Evolución 1...\n")

for phase_id in test_phase_ids:
    print(f"{'='*80}")
    print(f"FASE {phase_id}")
    print(f"{'='*80}")
    
    # Obtener eventos de la fase
    phase_events = get_events_by_possession_id(pro_phases, phase_id=phase_id, events_df=events)
    
    if phase_events is None or len(phase_events) == 0:
        print(f"⚠️ No se encontraron eventos para la fase {phase_id}\n")
        continue
    
    # Obtener información básica de la fase
    phase_info = pro_phases[pro_phases['Phase_ID'] == phase_id].iloc[0]
    print(f"Equipo: {phase_info['Team']}")
    print(f"Duración: {phase_info['Duration']:.2f}s")
    print(f"Eventos: {len(phase_events)}")
    print(f"Zona inicio: {phase_info['Start_Zone']} → Zona final: {phase_info['End_Zone']}")
    print(f"Resultado: {phase_info['Outcome']}\n")
    
    # Generar descripción
    print("📝 Generando descripción...")
    description = generate_description_v1(phase_events)
    
    print("\n" + "─"*80)
    print("DESCRIPCIÓN TÁCTICA (Evolución 1):")
    print("─"*80)
    print(description)
    print("\n")

🧪 Generando descripciones tácticas con Evolución 1...

FASE 4
Equipo: Home
Duración: 9.60s
Eventos: 6
Zona inicio: Iniciación → Zona final: Finalización
Resultado: Pérdida

📝 Generando descripción...

────────────────────────────────────────────────────────────────────────────────
DESCRIPCIÓN TÁCTICA (Evolución 1):
────────────────────────────────────────────────────────────────────────────────
El equipo *Home* inicia una construcción ofensiva desde su zona defensiva (x ≈ 0.32), con un pase corto de cabeza de **Player5** hacia **Player6** (zona central, interior izquierdo), manteniendo la orientación hacia la derecha. Tras una breve pausa, **Player6** proyecta hacia adelante con un pase horizontal a **Player10** (zona central, carril central, x ≈ 0.41), ampliando el ancho y buscando profundidad progresiva. Posteriormente, **Player10** ejecuta un pase largo y diagonal hacia **Player8**, que avanza desde la zona central hasta la zona ofensiva (x ≈ 0.56 → 0.86), ocupando el pasillo izquie

In [ ]:
def convert_to_coordinates(x_norm, y_norm, pitch_length=105, pitch_width=68):
    """
    Convierte coordenadas normalizadas (0-1) a coordenadas absolutas del campo.
    Parámetros:
        x_norm (float or pd.Series): Coordenada X normalizada (0-1, eje largo del campo).
        y_norm (float or pd.Series): Coordenada Y normalizada (0-1, eje ancho del campo).
        pitch_length (float): Longitud total del campo (por defecto 105 metros).
        pitch_width (float): Ancho total del campo (por defecto 68 metros).
    Retorna:
        (x_abs, y_abs): tupla de coordenadas absolutas en metros.
    """
    x_abs = x_norm * pitch_length
    y_abs = y_norm * pitch_width
    return x_abs, y_abs

In [56]:
x_min = 0 
x_max=105 
y_min=0 
y_max=68
# Colonne normalizzate (nomi tipici Metrica)
events['Start X'], events['Start Y'] = convert_to_coordinates(
    events['Start X'], events['Start Y'], pitch_length=x_max, pitch_width=y_max
)
events['End X'], events['End Y'] = convert_to_coordinates(
    events['End X'], events['End Y'], pitch_length=x_max, pitch_width=y_max
)

In [ ]:
# Seleccionar algunas secuencias interesantes para probar
test_phase_ids = [1, 5, 10]

print("🧪 Generando descripciones tácticas con Evolución 1...\n")

for phase_id in test_phase_ids:
    print(f"{'='*80}")
    print(f"FASE {phase_id}")
    print(f"{'='*80}")
    
    # Obtener eventos de la fase
    phase_events = get_events_by_possession_id(pro_phases, phase_id=phase_id, events_df=events)
    
    if phase_events is None or len(phase_events) == 0:
        print(f"⚠️ No se encontraron eventos para la fase {phase_id}\n")
        continue
    
    # Obtener información básica de la fase
    phase_info = pro_phases[pro_phases['Phase_ID'] == phase_id].iloc[0]
    print(f"Equipo: {phase_info['Team']}")
    print(f"Duración: {phase_info['Duration']:.2f}s")
    print(f"Eventos: {len(phase_events)}")
    print(f"Zona inicio: {phase_info['Start_Zone']} → Zona final: {phase_info['End_Zone']}")
    print(f"Resultado: {phase_info['Outcome']}\n")
    
    # Generar descripción
    print("📝 Generando descripción...")
    description = generate_description_v1(phase_events, x_min, x_max, y_min, y_max)
    
    print("\n" + "─"*80)
    print("DESCRIPCIÓN TÁCTICA (Evolución 1):")
    print("─"*80)
    print(description)
    print("\n")

🧪 Generando descripciones tácticas con Evolución 1...

FASE 1
Equipo: Away
Duración: 19.88s
Eventos: 15
Zona inicio: Creación → Zona final: Finalización
Resultado: Pérdida

📝 Generando descripción...

────────────────────────────────────────────────────────────────────────────────
DESCRIPCIÓN TÁCTICA (Evolución 1):
────────────────────────────────────────────────────────────────────────────────
La jugada inicia con un *kick-off* de *Away* en su propia zona defensiva (x ≈ 47), atacando de derecha a izquierda (x decreciente). Tras un pase corto (Player19 → Player21) hacia el carril derecho (x: 47.25 → 57.75), se produce una progresión vertical hacia el carril izquierdo ofensivo (Player21 → Player15, x: 57.75 → 60.90, y: 29.24 → 14.28), buscando el tercer tercio derecho. Tras una pausa (1.68 s), el balón regresa a Player19 en zona media (x: 47.25, y: 21.08), donde se inicia una segunda fase: pase transversal hacia el centro (Player19 → Player21, x: 47.25 → 51.45, y: 21.76 → 31.96), seguid

## 7. Evolución 2: Eventos + Documentación (RAG)

Generación de descripciones tácticas usando **eventos + documentación táctica** como contexto (RAG). Se recuperan los fragmentos más relevantes de la documentación según la secuencia de eventos y se incluyen en el prompt para enriquecer la descripción con terminología y conceptos del marco de análisis.

In [27]:
def retrieve_relevant_chunks(query, doc_chunks, doc_embeddings, embedding_model, top_k=3):
    """
    Recupera los top_k fragmentos de documentación más relevantes para una consulta
    usando similitud coseno entre embeddings.
    """
    query_emb = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    sims = cosine_similarity(query_emb, doc_embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]
    return [doc_chunks[i] for i in top_indices]


def generate_description_v2(events_list, top_k=3, x_min=0, x_max=105, y_min=0, y_max=68):
    """
    Genera una descripción táctica usando eventos + documentación (RAG).
    Recupera los chunk más relevantes de la documentación y los incluye en el prompt.
    """
    events_text = format_events_as_text(events_list)
    # La consulta para retrieval: resumen de la secuencia (tipos de eventos y contexto)
    query = events_text[:3000]  # truncar si es muy largo para el embedding
    relevant_chunks = retrieve_relevant_chunks(
        query, doc_chunks, doc_embeddings, embedding_model, top_k=top_k
    )
    context_parts = []
    for i, c in enumerate(relevant_chunks, 1):
        context_parts.append(f"--- Fragmento {i} ({c['title']}) ---\n{c['text']}")
    doc_context = "\n\n".join(context_parts)

    prompt = f"""Eres un analista táctico de fútbol profesional. A continuación se te proporciona:
1) Documentación de referencia sobre análisis táctico (zonas, fases ofensivas/defensivas, terminología).
2) Una secuencia de eventos de una jugada.

Usa la documentación para enriquecer tu descripción: emplea la terminología y los conceptos que apliquen (zonas 1/2/3, construcción baja, amplitudes, transiciones, presión, etc.). Si la jugada encaja en alguna fase descrita en la documentación, menciónalo.

=== DOCUMENTACIÓN DE REFERENCIA ===
{doc_context}

=== SECUENCIA DE EVENTOS ===
{events_text}

Instrucciones:
- Clasifica la tipología de la posesión (por ejemplo: construcción baja, desarrollo, ataque a la portería, transición ofensiva, transición defensiva, bloque medio, bloque bajo, primera presión).
- No agregues suposiciones, limítate estrictamente a lo que ves.
- Para cada evento, no tienes la posición de los demás jugadores, así que no sabes cuál es la disposición de los jugadores.
- La descripción debe contener únicamente la clasificación que has realizado y una breve descripción de cómo se desarrolla la acción a través de los eventos.
- Los datos proporcionados son datos de eventos, así que céntrate principalmente en el equipo que está llevando la posesión del balón.
Descripción táctica:"""


    description = generate_with_api(prompt, max_new_tokens=1000, temperature=0.7)
    return description.strip()


print("✅ Funciones generate_description_v2 y retrieve_relevant_chunks definidas")

✅ Funciones generate_description_v2 y retrieve_relevant_chunks definidas


### 7.1. Prueba de Evolución 2

Probamos la generación con RAG sobre las mismas secuencias que en Evolución 1 para comparar el uso de terminología táctica de la documentación.

In [28]:
print("🧪 Generando descripciones tácticas con Evolución 2 (RAG)...\n")

for phase_id in test_phase_ids:
    print(f"{'='*80}")
    print(f"FASE {phase_id}")
    print(f"{'='*80}")

    phase_events = get_events_by_possession_id(pro_phases, phase_id=phase_id, events_df=events)
    if phase_events is None or len(phase_events) == 0:
        print(f"⚠️ No se encontraron eventos para la fase {phase_id}\n")
        continue

    phase_info = pro_phases[pro_phases['Phase_ID'] == phase_id].iloc[0]
    print(f"Equipo: {phase_info['Team']} | Duración: {phase_info['Duration']:.2f}s | Eventos: {len(phase_events)}")
    print(f"Zona: {phase_info['Start_Zone']} → {phase_info['End_Zone']} | Resultado: {phase_info['Outcome']}\n")

    print("📝 Generando descripción (v2 con documentación)...")
    description_v2 = generate_description_v2(phase_events, top_k=3, x_min=0, x_max=1, y_min=0, y_max=1)

    print("\n" + "─"*80)
    print("DESCRIPCIÓN TÁCTICA (Evolución 2 - RAG):")
    print("─"*80)
    print(description_v2)
    print("\n")

🧪 Generando descripciones tácticas con Evolución 2 (RAG)...

FASE 4
Equipo: Home | Duración: 9.60s | Eventos: 6
Zona: Iniciación → Finalización | Resultado: Pérdida

📝 Generando descripción (v2 con documentación)...

────────────────────────────────────────────────────────────────────────────────
DESCRIPCIÓN TÁCTICA (Evolución 2 - RAG):
────────────────────────────────────────────────────────────────────────────────
**Clasificación:** *Construcción baja → Desarrollo (Zona 2) → Transición ofensiva*  

**Descripción táctica:**  
El equipo *Home* inicia una posesión desde su zona defensiva (x ≈ 0.32 → 0.31), lo que indica **construcción baja**. El balón avanza por el carril izquierdo (Player5 → Player6 → Player10), con pases progresivos hacia el tercio medio (x sube de 0.32 a 0.41), posicionándose en la **Zona 2 propia** (bloque medio rival no observable, pero el avance sugiere superación del primer filtro). El siguiente pase (Player10 → Player8) es progresivo y dirigido a la **Zona de Re